# Step 5: Data Preprocessing, Anomaly Cleansing, and One-Hot Encoding

### 1 - DATASET INGESTION & OUTLIER ANOMALY FILTERING

* **The Pipeline Phase:** Data Preprocessing & Anomaly Cleansing.
* **The Problematics:** Real-world hospitality datasets suffer from typos, test entries, and data pipeline artifacts. These include booking records with 0 guests (which are database ghosts), negative or extremely bloated room prices (Average Daily Rate - ADR), and rare category values (e.g., country codes that appear only once, which unnecessarily bloat our encoded matrix).
* **Our Solution:** We load the checkpoint CSV, dynamically audit and remove logical paradoxes, restrict ADR to positive values below a statistical 99.9th percentile ceiling, and collapse rare countries into a structured 'Other' dimension.

In [ ]:
import pandas as pd
import numpy as np

# 1. LOAD THE CHECKPOINT CSV
df = pd.read_csv('bookings_with_holiday_features.csv')
print(f"Initial Row Count: {df.shape[0]:,}")

# 2. REMOVE GHOST BOOKINGS (Zero occupants across all age categories)
occupant_mask = (df['adults'] + df['children'] + df['babies']) > 0
df = df[occupant_mask].copy()

# 3. ADR CLEANING AND TYPO TRUNCATION
# Drop negative or zero ADR bookings (free rooms or financial system typos)
df = df[df['adr'] > 0].copy()

# Dynamically calculate the 99.9th percentile to filter out extreme typos (e.g. $5000/night)
adr_ceiling = df['adr'].quantile(0.999)
df = df[df['adr'] <= adr_ceiling].copy()

# 4. GROUP RARE COUNTRIES TO PREVENT COLUMN BLOAT
# If a country appears fewer than 10 times, group it into 'Other' so we don't one-hot encode it
country_counts = df['country'].value_counts(dropna=False)
rare_countries = country_counts[country_counts < 10].index
df['country'] = df['country'].replace(rare_countries, 'Other')

print(f"Cleaned Row Count: {df.shape[0]:,}")
print(f"ADR Statistical Ceiling Applied: ${adr_ceiling:.2f}")

#### Preprocessing Logic & Code Explanations:
* **`df['country'].value_counts()`**: Evaluates the statistical distribution of customer origins in the dataset. Grouping country codes that occur fewer than 10 times into `'Other'` protects against over-parameterization during one-hot encoding.
* **`df['adr'].quantile(0.999)`**: Employs percentile-based truncation to establish a dynamic, data-driven boundary for pricing. It preserves expensive suite reservations while removing typographical errors.

### 2 - EXPLICIT NULL ALLOCATION & ONE-HOT ENCODING

* **The Pipeline Phase:** Missing Value Allocation & One-Hot Encoding.
* **The Problematics:** Missing identifiers in system tables (like `agent` and `company`) will throw errors when loaded into scikit-learn. Furthermore, categorical columns like `deposit_type` and `market_segment` cannot be used directly. To prevent data leakage, explicit date anchors (`booking_date`, `arrival_date`, `departure_date`) must be removed.
* **Our Solution:** We fill numeric null IDs with `-1`, flag empty countries, perform stable one-hot encoding, cast the outputs into clean integers, and drop raw date columns.

In [ ]:
# 1. SEPARATE LABELS & EXCLUDE RAW CHRONOLOGICAL DATE KEYS
y = df['is_canceled'].copy()
features_to_drop = ['is_canceled', 'booking_date', 'arrival_date', 'departure_date']
X_raw = df.drop(columns=features_to_drop)

# 2. EXPLICIT MISSING VALUE IMPUTATION
# For categorical items, fill blanks with a structural string indicator
if 'country' in X_raw.columns:
    X_raw['country'] = X_raw['country'].fillna('Unknown')
if 'distribution_channel' in X_raw.columns:
    X_raw['distribution_channel'] = X_raw['distribution_channel'].fillna('Unknown')

# For structural ID systems, treat empty cells as a specific missing index value (-1)
if 'agent' in X_raw.columns:
    X_raw['agent'] = X_raw['agent'].fillna(-1).astype('int64')
if 'company' in X_raw.columns:
    X_raw['company'] = X_raw['company'].fillna(-1).astype('int64')

# 3. ONE-HOT ENCODING THE CATEGORICAL WALL
categorical_columns = [
    'hotel', 'deposit_type', 'market_segment', 'meal', 
    'customer_type', 'distribution_channel', 'reserved_room_type', 'country'
]

# Use pd.get_dummies to convert text categories into independent numeric columns
X = pd.get_dummies(X_raw, columns=categorical_columns, drop_first=True)

# Convert boolean arrays (True/False) into clean integer flags (1/0)
bool_columns = X.select_dtypes(include=['bool']).columns
X[bool_columns] = X[bool_columns].astype(int)

print("--- PREPROCESSING & ENCODING SUCCESS ---")
print(f"Target Array shape (y): {y.shape}")
print(f"Feature Matrix shape (X): {X.shape}")
print(f"Total columns generated for ML processing: {len(X.columns)}")

#### Preprocessing Logic & Code Explanations:
* **`drop_first=True`**: Removes the first category column generated for each categorical variable. For example, a category with three options is encoded into two columns. This prevents collinearity, which can destabilize regression models.
* **`X_raw.drop(columns=features_to_drop)`**: Drops the raw date stamps. By keeping only the derived attributes (like monthly demand, week numbers, and holiday types), we protect the model against temporal leakage.